# SPATIAL INTELLIGENCE - PART 1

In [1]:
# topologicpy is pip-installed; no sys.path needed.

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

C:\PROJECTS\GRAPH-ML-DOCUMENTS\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.


The version that you are using (0.9.36) is OLDER than the latest version (0.9.52) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [5]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [6]:
gallery = Topology.ByBREPPath(r"C:/PROJECTS/GRAPH-ML-DOCUMENTS/FINAL/floorplate-4-182-180.brep")


## 6. Show the geometry

In [7]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [8]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

# Coarse grid step in meters. Smaller -> more cells -> slower analysis.
STEP = 2.5
def frange(stop, step):
    out, x = [], 0.0
    while x <= stop + step:
        out.append(round(x, 3)); x += step
    return out
uRange = frange(width, STEP)
vRange = frange(length, STEP)

# Slicing a multi-face cluster at once does not subdivide, so build the grid per face.
grid_parts = [Grid.EdgesByDistances(f, clip=True, uRange=uRange, vRange=vRange)
              for f in Topology.Faces(gallery)]
grid = Cluster.ByTopologies(grid_parts)

## 8. Show the geometry and the grid

In [9]:
Topology.Show(gallery, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [10]:
# Slice each face (wing) with its own grid -> a Shell of connected cells per wing.
# Combining the wing SHELLS (not loose faces) preserves cell adjacency for the graph.
wing_shells = []
for f in Topology.Faces(gallery):
    gf = Grid.EdgesByDistances(f, clip=True, uRange=uRange, vRange=vRange)
    wing_shells.append(Topology.Slice(f, gf))
shell = Cluster.ByTopologies(wing_shells)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)
print("Number of cells:", len(faces))

Number of cells: 381


## 10. Show the resulting shell

In [11]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 11. Derive navigation and analysis graphs from the shell

In [12]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 12. Derive and store the analysis graph vertices

In [13]:
g_verts = Graph.Vertices(analysis_graph)

## 13. Show the analysis graph

In [14]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## 14. Spatial Intelligence through Graph Analysis

### a. Minimum Spanning Tree
MST is very time consuming to compute on a large grid graph. Here we will demonstrate it on a simple graph

In [15]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g2 = Graph.MinimumSpanningTree(g2)   # g2 reduced to its minimum spanning tree
Topology.Show(g1, g2,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [16]:
dn1 = Graph.Density(g1)
dn2 = Graph.Density(g2)
print("Density 1 (full prism graph):", dn1)
print("Density 2 (minimum spanning tree):", dn2)

Density 1 (full prism graph): 0.42857142857142855
Density 2 (minimum spanning tree): 0.25


In [17]:
dr1 = Graph.Diameter(g1)
dr2 = Graph.Diameter(g2)
print("Diameter 1 (full prism graph):", dr1)
print("Diameter 2 (minimum spanning tree):", dr2)

Diameter 1 (full prism graph): 3
Diameter 2 (minimum spanning tree): 5


### Community Detection
Community detection groups densely-connected cells into clusters. Cells in the same community are coloured alike, revealing the natural spatial zones of the floor plate.

In [18]:
# Detect communities on the analysis graph and colour each cell by its community.
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")
g_verts = Graph.Vertices(analysis_graph)
n_comm = len(set(Dictionary.ValueAtKey(Topology.Dictionary(v), "community") for v in g_verts))
print("Number of communities:", n_comm)

reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Number of communities: 18


### Degree Centrality (community graph)

**Bin By Dictionary Key**
* Detect communities, then use the community number to separate the cells into bins.
* Derive the outer boundary (perimeter) of each face group and make a face from it.

In [19]:
# Community detection (needed to bin the cells), then transfer to the shell faces
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")
g_verts = Graph.Vertices(analysis_graph)
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
fg_keys = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    if temp_shell is None:
        continue
    eb = Shell.ExternalBoundary(temp_shell)
    if eb is None:
        continue
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    if eb is not None:
        face_groups.append(eb); fg_keys.append(key)

print("Number of face groups:", len(face_groups))

Number of face groups: 18


Show the result

In [20]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new shell from the new faces

*The building has separate blocks, so the community faces are grouped per block into shells and clustered — a single shell across disconnected blocks is not possible.*

In [21]:
from collections import defaultdict
wings = Topology.Faces(gallery)
groups_by_wing = defaultdict(list)
for key, fg in zip(fg_keys, face_groups):
    c = Topology.Centroid(fg)
    wi = next((i for i, wf in enumerate(wings) if Vertex.IsInternal(c, wf)), None)
    if wi is None:
        wi = min(range(len(wings)), key=lambda i: Vertex.Distance(c, Topology.Centroid(wings[i])))
    groups_by_wing[wi].append(fg)
region_shells = [Shell.ByFaces(v) for v in groups_by_wing.values()]
region_shells = [s for s in region_shells if s is not None]
new_shell = Cluster.ByTopologies(region_shells)

Show the result

In [22]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new graph from the new shell

In [23]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

Show the result

In [24]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Compute degree centralities

In [25]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)
new_verts = Graph.Vertices(new_graph)
print("Number of community nodes:", len(new_verts))

Number of community nodes: 18


Transfer / interpolate values from the new graph vertices to the original graph vertices

In [26]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

Derive the colour of each vertex based on the interpolated value

In [27]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

Transfer the information from the graph vertices to the faces of the original shell

In [28]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

Show the result

In [29]:
Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              vertexSizeKey="size",
              vertexColorKey="dc_color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### b. Shortest Path (Use navigation graph)

In [30]:
import time

# The plate has two disconnected wings, so a corner-to-corner path may not exist.
# Pick start/end inside the LARGEST connected component so a path is guaranteed.
components = Graph.ConnectedComponents(navigation_graph)
main = max(components, key=lambda c: len(Graph.Vertices(c) or []))
cvs = Graph.Vertices(main)
start_vertex = min(cvs, key=lambda v: Vertex.X(v) + Vertex.Y(v))   # one end of the wing
end_vertex   = max(cvs, key=lambda v: Vertex.X(v) + Vertex.Y(v))   # opposite end

crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
try:
    straight_path = Wire.Straighten(shortest_path, host=gallery)
except Exception:
    straight_path = shortest_path

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

Shortest Path Duration: 0.13 seconds


Original Shortest Path Length: 50.26
Straightened Shortened Path Length: 46.03


In [31]:
Topology.Show(gallery, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [32]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [33]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [34]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [35]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [36]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [37]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### e. Visibility Graph Analysis (VGA)
Following the HW02 isovist approach: from a grid of viewpoints we cast an **isovist** — the
polygon of everything visible from that point, blocked by the wing boundary and any interior
obstacles. The isovists are overlaid on the floor plate. Where many isovists overlap the space
reads lighter (a visually integrated core); secluded pockets stay darker. Computed per wing.

In [38]:
# Visibility Graph Analysis via isovists (HW02 method)
import time

faces = Topology.Faces(shell)
g_verts = Graph.Vertices(analysis_graph)          # dense points = cell centroids
wings = Topology.Faces(gallery)

# Coarse grid of viewpoints (one isovist per cell is far too slow), HW02-style.
ISO_STEP = 5.0
uR_iso = frange(width, ISO_STEP)
vR_iso = frange(length, ISO_STEP)

isovists = []
new_verts = []
n_list = []
t = time.time()
for wing in wings:
    grid1 = Grid.EdgesByDistances(wing, clip=True, uRange=uR_iso, vRange=vR_iso)
    iso_pts = [v for v in Topology.Vertices(grid1) if Vertex.IsInternal(v, wing)]
    dense_in_wing = [v for v in g_verts if Vertex.IsInternal(v, wing)]
    for v in iso_pts:
        iso = Face.Isovist(wing, v, silent=True)   # holes act as obstacles
        if iso:
            isovists.append(iso)
            b_list = Vertex.IsInternal2D(dense_in_wing, iso)
            n = len([b for b in b_list if b])
            n_list.append(n)
            d = Dictionary.ByKeyValue("visibility", n)
            v = Topology.SetDictionary(v, d)
            new_verts.append(v)

print("Number of valid isovists:", len(isovists), "computed in %.1fs" % (time.time() - t))

# Overlay the isovists on the floor plate (grey, semi-transparent) - HW02 look.
Topology.Show(gallery, isovists,
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Number of valid isovists: 240 computed in 773.5s


### Compute the visibility of each viewpoint and colour the cells
Each viewpoint's **visibility** is how many of the dense grid cells fall inside its isovist.
These values are interpolated onto every cell and coloured on a thermal scale: bright cells are
visually open / well-connected, dark cells are visually secluded.

In [39]:
# Interpolate the visibility value from the coarse viewpoints onto every cell centroid.
for v in g_verts:
    Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="visibility")

# Colour each cell by its (interpolated) visibility, thermal scale.
# Normalise over the actual interpolated cell values so the gradient spans the full range.
vals = [Dictionary.ValueAtKey(Topology.Dictionary(v), "visibility") for v in g_verts]
vals = [x for x in vals if x is not None]
minValue = min(vals)
maxValue = max(vals)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    if vb is None:
        continue
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    v = Topology.SetDictionary(v, d)

# Transfer the colours from the graph vertices to the shell faces.
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

In [40]:
status = Topology.ExportToBREP(gallery, path=r"C:/PROJECTS/GRAPH-ML-DOCUMENTS/FINAL/floorplate-4-182-180.brep", overwrite=True)
print(status)

True
